# Πολυτροπική Ανάλυση Μουσικών Δεδομένων — Multimodal Music Analysis

**Μάθημα:** Τεχνικές Εξόρυξης Δεδομένων

**[ΟΝΟΜΑ ΦΟΙΤΗΤΗ] — sdi2200160**

## Notebook Setup Requirements

This notebook is self-contained in terms of Python implementation: the Part A data preparation, EDA helpers, and Part B classification helpers are defined inside the notebook. To rerun it, the environment still needs the Python packages installed by the first code cell.

The notebook detects the project directory automatically when it is launched either from this directory or from the repository root. In this note, all paths are relative to the project directory, i.e. the directory containing this notebook.

Expected data directory:

```text
data/
```

For a full rebuild from raw data, place these files in `data/`:

| File | Purpose |
| ---- | ------- |
| `id_mfcc_stats.tsv.bz2` | MFCC audio statistics |
| `processed_lyrics.tar.gz` | Lyrics text files |
| `id_genres.csv` | Comma-separated genre labels |
| `id_tags.csv` | Comma-separated user tags |
| `id_information.csv` | Song metadata |

If the cached Part A files are already present in `data/`, the notebook will load them instead of rebuilding embeddings:

```text
dataset.5.csv
dataset.5.genres.parquet
dataset.5.tags.parquet
dataset.5.audio.parquet
dataset.5.lyrics.parquet
```

Part B writes and reuses results in:

```text
results/
```

If `classification_metrics.csv`, `classification_cv_summary.csv`, and `f1_macro_comparison.png` already exist in `results/`, the final Part B cell displays the cached outputs instead of recomputing the full experiment. Set `FORCE_PART_B_RUN = True` in that cell to rerun the full classification pipeline.

In [ ]:
%pip install -q pandas numpy pyarrow joblib torch sentence-transformers transformers scikit-learn matplotlib seaborn wordcloud nltk rich

In [ ]:
from __future__ import annotations

import functools
import logging
import math
import pathlib
import sys
import tarfile
import typing
from dataclasses import dataclass

import matplotlib.axes
import matplotlib.figure
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rich.progress
import seaborn as sns
import sentence_transformers
import torch
from nltk.corpus import stopwords
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from wordcloud import WordCloud

import nltk
nltk.download("stopwords", quiet=True)

%matplotlib inline

# === Configuration ===

CANDIDATE_PROJECT_DIRS = [
	pathlib.Path.cwd(),
	pathlib.Path.cwd() / "project",
	pathlib.Path.cwd() / "data-mining" / "project",
]
PROJECT_DIR = next(
	(candidate for candidate in CANDIDATE_PROJECT_DIRS if (candidate / "src").exists()),
	pathlib.Path.cwd(),
)

if str(PROJECT_DIR) not in sys.path:
	sys.path.insert(0, str(PROJECT_DIR))

DATA_DIR = PROJECT_DIR / "data"
RESULTS_DIR = PROJECT_DIR / "results"
K = 5

## 1. Συλλογή Δεδομένων — Data Collection

We are given five raw data files (all tab-separated, keyed by Song ID):

| File | Description |
| ---- | ----------- |
| `id_mfcc_stats.tsv.bz2` | MFCC means (13) and covariance matrix (91 entries) per song |
| `processed_lyrics.tar.gz` | Preprocessed/stemmed lyrics, one text file per song |
| `id_genres.csv` | Comma-separated genre labels per song (multilabel) |
| `id_tags.csv` | Comma-separated user tags per song |
| `id_information.csv` | Song metadata: artist, song name, album |

### Design Choices

> **Multilabel genres**: The assignment suggests filtering to ~5,000–10,000 songs by selecting the top-5 genres as a *multiclass* problem (one genre per song). We instead treat genres as **multilabel** — a song tagged `"rock,indie rock"` belongs to *both* genres. This retains far more songs (~54,000 after intersection) and better reflects the reality that music rarely fits a single category. This choice propagates through the entire pipeline: encodings are binary vectors, not one-hot; evaluation metrics must account for multilabel structure.

> **Full feature intersection**: Rather than intersecting only the minimum three files (audio, lyrics, genre), we include **all five sources** — adding tags and song metadata. Songs with any missing value in any source are dropped. This gives us a richer representation at the cost of a slightly smaller intersection.

In [ ]:
type BinaryDataFrame = pd.DataFrame


class MusicSeries(pd.Series):

	@property
	def _constructor(self) -> type[typing.Self]:
		return self.__class__

	@classmethod
	def from_csv(cls, path: str | pathlib.Path) -> typing.Self:
		return cls(
			pd.read_csv(path,
				sep="\t",
				index_col=0,
				low_memory=False,
			).squeeze()
		)

	@classmethod
	def from_tar(cls, path: str | pathlib.Path) -> typing.Self:
		records: dict[str, str] = {}

		with tarfile.open(path, "r:gz") as archive:
			for member in archive.getmembers():
				if not member.isfile():
					continue

				song_id = pathlib.Path(member.name).stem
				file = archive.extractfile(member)

				if file is not None:
					records[song_id] = file.read().decode("utf-8",
						errors="replace",
					)

		return cls(
			pd.Series(records, name="lyrics")
		)

	@functools.cached_property
	def encoding(self) -> BinaryDataFrame:
		return self.str.get_dummies(sep=",")

	def mask(self, genres: typing.Iterable[str], multi: bool = True) -> pd.Series:
		if multi: return self.encoding[genres].any(axis="columns")
		else: return self.isin(genres)

	def distribution(self, multi: bool = True) -> pd.Series:
		if multi: return self.encoding.sum(axis="index")
		else: return self.value_counts()

	def top_labels(self, k: int, multi: bool = True) -> pd.Index:
		if multi: return self.distribution(multi=True).sort_values(ascending=False).head(k).index
		else: return self.distribution(multi=False).head(k).index

	def top(self, k: int, multi: bool = True) -> pd.Series:
		return self[self.mask(self.top_labels(k, multi=multi), multi=multi)]


class MusicDataFrame(pd.DataFrame):

	@property
	def _constructor(self) -> type[typing.Self]:
		return self.__class__

	@classmethod
	def from_csv(cls, path: str | pathlib.Path) -> typing.Self:
		return cls(
			pd.read_csv(path,
				sep="\t",
				index_col=0,
				low_memory=False,
			)
		)

	def intersection(self, *attributes: pd.Series) -> pd.DataFrame:
		indices = self.index.intersection(
			pd.Index(set.intersection(*(set(attribute.index) for attribute in attributes)))
		)
		combined = pd.concat([attribute.loc[indices] for attribute in attributes], axis="columns")
		mask = combined.notna().all(axis="columns") \
			& combined.astype(str).apply(lambda column: column.str.strip().ne("")).all(axis="columns")

		return pd.concat([combined.loc[mask], self.loc[indices].loc[mask]], axis="columns")

In [ ]:
dataset_path = DATA_DIR / f"dataset.{K}.csv"

if dataset_path.exists():
	dataset = pd.read_csv(dataset_path, index_col=0)
	print(f"Loaded cached dataset from {dataset_path}")
else:
	lyrics = MusicSeries.from_tar(DATA_DIR / "processed_lyrics.tar.gz")
	genres = MusicSeries.from_csv(DATA_DIR / "id_genres.csv")
	tags = MusicSeries.from_csv(DATA_DIR / "id_tags.csv")
	info = MusicDataFrame.from_csv(DATA_DIR / "id_information.csv")
	audio_stats = MusicDataFrame.from_csv(DATA_DIR / "id_mfcc_stats.tsv.bz2")

	dataset = audio_stats.intersection(
		genres.top(K), tags,
		*[info[column] for column in info.columns],
		lyrics,
	)
	dataset.to_csv(dataset_path)
	print(f"Built and cached dataset to {dataset_path}")

print(f"Dataset shape: {dataset.shape}")
dataset.head()

## 2. Εξαγωγή Χαρακτηριστικών & Embeddings — Feature Extraction

We produce four separate embedding matrices, all indexed by Song ID:

### Text Embeddings (Lyrics)

> **Deviation from assignment**: The assignment suggests training Word2Vec or Doc2Vec on the lyrics corpus. We instead use a **pre-trained Sentence Transformer** (`all-MiniLM-L6-v2`), which maps each song's lyrics to a **384-dimensional** dense vector. This is a deliberate choice: the lyrics have already been stemmed and preprocessed, which removes inflection — a pre-trained transformer with BPE subword tokenization handles this gracefully, while training Word2Vec on stemmed text would yield a weaker vocabulary. The model and embedding dimensionality are **indicative**, not optimized — a larger model (e.g. `all-mpnet-base-v2` at 768 dims) could improve downstream quality.

### Audio Embeddings (MFCC)

> **Deviation from assignment**: Rather than PCA (the default suggestion), we train a **PyTorch autoencoder** (the bonus approach) to compress the 104-dimensional MFCC feature vector (13 means + 91 covariance entries) down to a bottleneck of **⌈√104⌉ = 11 dimensions**. The autoencoder uses a single hidden layer with SiLU activation. The bottleneck size is a simple heuristic — **not tuned** — and could be increased for better reconstruction fidelity.

### Genre & Tag Encodings

Genres and tags are both comma-separated multilabel strings. We apply binary one-hot encoding via `str.get_dummies(sep=",")`, producing a binary matrix where each column is a genre/tag and each row indicates presence (1) or absence (0).

In [ ]:
if torch.cuda.is_available():
	torch.set_default_device("cuda")


class AudioAutoencoder(torch.nn.Module):

	def __init__(self, input_dim: int, bottleneck: int | None = None) -> None:
		super().__init__()

		self.bottleneck = bottleneck or math.isqrt(input_dim) + 1

		self.encoder = torch.nn.Sequential(
			torch.nn.Linear(input_dim, self.bottleneck),
			torch.nn.SiLU(),
		)
		self.decoder = torch.nn.Sequential(
			torch.nn.Linear(self.bottleneck, input_dim),
		)

	def forward(self, x: torch.Tensor) -> torch.Tensor:
		return self.decoder(self.encoder(x))

	@staticmethod
	@torch.no_grad
	def normalize(data: torch.Tensor) -> torch.Tensor:
		return (data - data.mean(dim=0)) / data.std(dim=0)

	def compile(self,
		optimizer: torch.optim.Optimizer | None = None,
		loss_fn: torch.nn.MSELoss | None = None,
		lr: float = 1e-3,
	) -> None:
		self.optimizer = optimizer or torch.optim.Adam(self.parameters(), lr=lr)
		self.loss_fn = loss_fn or torch.nn.MSELoss()

	def fit(self, data: torch.Tensor,
		epochs: int = 1,
		batch_size: int | None = None,
	) -> None:
		dataset = torch.utils.data.TensorDataset(self.normalize(data))
		loader = torch.utils.data.DataLoader(dataset,
			batch_size=batch_size or math.isqrt(len(data)) + 1,
			shuffle=True,
			generator=torch.Generator(device=data.device),
		)

		self.train()

		with rich.progress.Progress(
			rich.progress.TextColumn("[bold blue]{task.description}"),
			rich.progress.BarColumn(),
			rich.progress.MofNCompleteColumn(),
			rich.progress.TextColumn("loss: {task.fields[loss]:.4f}"),
			rich.progress.TimeRemainingColumn(),
			rich.progress.TimeElapsedColumn(),
			refresh_per_second=60,
		) as progress:
			epoch_task = progress.add_task("epoch", total=epochs, loss=0.)
			batch_task = progress.add_task("batch", total=len(loader), loss=0.)

			cumulative_loss = 0.

			for epoch in range(epochs):
				total_loss = 0.
				samples = 0

				progress.reset(batch_task, total=len(loader))

				for batch, in loader:
					loss = self.loss_fn(self(batch), batch); self.optimizer.zero_grad()
					loss.backward(); self.optimizer.step()

					total_loss += loss.item() * batch.size(0); samples += batch.size(0)
					progress.update(batch_task, advance=1, loss=total_loss / samples)

				cumulative_loss += total_loss / len(data)
				progress.update(epoch_task, advance=1, loss=cumulative_loss / (epoch + 1))

			progress.remove_task(batch_task); print()

		self.eval()

	@torch.no_grad
	def evaluate(self, data: torch.Tensor) -> float:
		normalized = self.normalize(data)
		return self.loss_fn(self(normalized), normalized).item()

	@torch.no_grad
	def encode(self, data: torch.Tensor) -> torch.Tensor:
		normalized = self.normalize(data)
		return self.encoder(normalized)


def encode_genres(genres: pd.Series) -> pd.DataFrame:
	return genres.str.get_dummies(sep=",")


def embed_audio(features: pd.DataFrame, epochs: int = 1) -> pd.DataFrame:
	tensor = torch.tensor(features.values, dtype=torch.float32)

	model = AudioAutoencoder(len(features.columns))
	model.compile()
	model.fit(tensor, epochs)
	embeddings = model.encode(tensor).numpy(force=True)

	return pd.DataFrame(embeddings,
		index=features.index,
		columns=[f"audio_{i:03d}" for i in range(model.bottleneck)],
	)


def embed_lyrics(lyrics: pd.Series, model_name: str = "all-MiniLM-L6-v2") -> pd.DataFrame:
	model = sentence_transformers.SentenceTransformer(model_name)
	embeddings = model.encode(lyrics.tolist())

	return pd.DataFrame(
		embeddings,
		index=lyrics.index,
		columns=[f"lyric_{i:03d}" for i in range(embeddings.shape[1])],
	)

In [ ]:
EPOCHS = 256

genres_path = DATA_DIR / f"dataset.{K}.genres.parquet"
audio_path  = DATA_DIR / f"dataset.{K}.audio.parquet"
lyrics_path = DATA_DIR / f"dataset.{K}.lyrics.parquet"
tags_path   = DATA_DIR / f"dataset.{K}.tags.parquet"

if all(p.exists() for p in (genres_path, audio_path, lyrics_path, tags_path)):
	genres_enc = pd.read_parquet(genres_path)
	audio_emb  = pd.read_parquet(audio_path)
	lyrics_emb = pd.read_parquet(lyrics_path)
	tags_enc   = pd.read_parquet(tags_path)
	print("Loaded cached embeddings.")
else:
	genres_enc = encode_genres(dataset["genres"])
	audio_emb  = embed_audio(
		dataset[[c for c in dataset.columns if c.startswith(("MFCC", "cov_"))]],
		EPOCHS,
	)
	lyrics_emb = embed_lyrics(dataset["lyrics"])
	tags_enc   = encode_genres(dataset["tags"])

	genres_enc.to_parquet(genres_path)
	audio_emb.to_parquet(audio_path)
	lyrics_emb.to_parquet(lyrics_path)
	tags_enc.to_parquet(tags_path)
	print("Generated and cached embeddings.")

print(f"Genre encodings:  {genres_enc.shape}")
print(f"Audio embeddings: {audio_emb.shape}")
print(f"Lyric embeddings: {lyrics_emb.shape}")
print(f"Tag encodings:	{tags_enc.shape}")

## 3. Οπτικοποίηση και Ανάλυση — Exploratory Data Analysis (EDA)

With the dataset loaded and all embeddings computed, we now explore the data through a series of visualizations. Each plot is produced by a standalone function that accepts an `EDAData` container holding the dataset and all embedding matrices.

In [ ]:
@dataclass
class EDAData:
	dataset: pd.DataFrame
	genres_enc: pd.DataFrame
	tags_enc: pd.DataFrame
	audio_emb: pd.DataFrame
	lyrics_emb: pd.DataFrame


data = EDAData(
	dataset=dataset,
	genres_enc=genres_enc,
	tags_enc=tags_enc,
	audio_emb=audio_emb,
	lyrics_emb=lyrics_emb,
)

### 3.1 Word Clouds by Genre

To pick the two genres for word cloud comparison, we automatically detect the **most different pair** among the top-5 genres. "Difference" is measured by cosine distance between genre-level tag profile vectors — i.e., for each genre we sum the binary tag vectors of all its songs (normalized), then find the pair with the lowest cosine similarity. This ensures the word clouds are maximally contrastive.

In [ ]:
def top_genre_names(data: EDAData, k: int = 5) -> list[str]:
	return data.genres_enc.sum().sort_values(ascending=False).head(k).index.tolist()


def most_different_genres(data: EDAData, genres: list[str]) -> tuple[str, str]:
	profiles = {}

	for genre in genres:
		mask = data.genres_enc[genre].astype(bool)
		profiles[genre] = data.tags_enc.loc[mask].sum().values.astype(float)
		profiles[genre] /= profiles[genre].sum() or 1.

	worst_sim = 1.
	pair = (genres[0], genres[1])

	for i, g1 in enumerate(genres):
		for g2 in genres[i + 1:]:
			sim = cosine_similarity(
				profiles[g1].reshape(1, -1),
				profiles[g2].reshape(1, -1),
			)[0, 0]

			if sim < worst_sim:
				worst_sim = sim
				pair = (g1, g2)

	return pair


def plot_wordcloud(data: EDAData, genre: str,
	ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
	mask = data.genres_enc[genre].astype(bool)
	tag_freq = data.tags_enc.loc[mask].sum().to_dict()

	cloud = WordCloud(
		width=800, height=400,
		background_color="white",
	).generate_from_frequencies(tag_freq)

	if ax is None: fig, ax = plt.subplots(figsize=(10, 5))
	else: fig = ax.figure

	ax.imshow(cloud, interpolation="bilinear")
	ax.set_title(f"Tag Word Cloud: {genre}")
	ax.axis("off")

	return fig


top_genres = top_genre_names(data, K)
g1, g2 = most_different_genres(data, top_genres)

print(f"Top {K} genres: {top_genres}")
print(f"Most different pair: {g1} vs {g2}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))
plot_wordcloud(data, g1, ax=ax1)
plot_wordcloud(data, g2, ax=ax2)
fig.tight_layout()
plt.show()

### 3.2 Top Tags

A simple bar chart of the 10 most frequent user-generated tags across the entire dataset. Tags are crowd-sourced descriptors and often overlap with (but are not identical to) genre labels — they capture listener perception rather than editorial taxonomy.

In [ ]:
def plot_top_tags(data: EDAData, n: int = 10,
	ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
	counts = data.tags_enc.sum().sort_values(ascending=True).tail(n)

	if ax is None: fig, ax = plt.subplots(figsize=(10, 6))
	else: fig = ax.figure

	ax.barh(counts.index, counts.values)
	ax.set_xlabel("Frequency")
	ax.set_title(f"Top {n} Most Frequent Tags")

	return fig


plot_top_tags(data)
plt.show()

### 3.3 Dimensionality Reduction — t-SNE

We apply t-SNE to project both audio and lyrics embeddings down to 2D, then plot each song as a point colored by its genre(s). Since genres are multilabel, a song may appear under multiple genre layers — we use low alpha (0.1) to handle overlap.

The side-by-side comparison reveals which modality separates genres more cleanly. We expect audio features to produce tighter clusters for genres with distinctive sonic profiles (e.g. electronic, hip-hop), while lyrics embeddings may better distinguish genres that share similar sound but differ thematically.

> **Note**: t-SNE on ~54k samples is CPU-bound via scikit-learn and takes a few minutes. Results are computed live.

In [ ]:
def reduce_tsne(embeddings: pd.DataFrame,
	n_components: int = 2,
	**kwargs,
) -> np.ndarray:
	return TSNE(
		n_components=n_components,
		random_state=42,
		**kwargs,
	).fit_transform(embeddings.values)


def plot_scatter_by_genre(data: EDAData,
	modality: str,
	coords_2d: np.ndarray,
	top_k: int = 5,
	ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
	if ax is None: fig, ax = plt.subplots(figsize=(10, 8))
	else: fig = ax.figure

	genres = top_genre_names(data, top_k)

	for genre in genres:
		mask = data.genres_enc[genre].astype(bool).values
		ax.scatter(
			coords_2d[mask, 0],
			coords_2d[mask, 1],
			label=genre, alpha=0.1, s=5,
		)

	ax.set_title(f"t-SNE: {modality.capitalize()} Embeddings by Genre")
	ax.legend(markerscale=5)

	return fig


audio_2d = reduce_tsne(data.audio_emb)
lyrics_2d = reduce_tsne(data.lyrics_emb)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))
plot_scatter_by_genre(data, "audio", audio_2d, top_k=K, ax=ax1)
plot_scatter_by_genre(data, "lyrics", lyrics_2d, top_k=K, ax=ax2)
fig.suptitle("Audio vs Text Embeddings — Genre Separation Comparison")
fig.tight_layout()
plt.show()

### 3.4 Genre Variety per Song

Music is complex — a song might be pure "rock" or straddle "rock, indie rock, alternative rock". This histogram shows the distribution of how many genres each song belongs to, illustrating the multilabel nature of the dataset. Most songs have 1–3 genres, but some belong to many more.

In [ ]:
def plot_genre_count_histogram(data: EDAData,
	ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
	genre_counts = data.dataset["genres"].str.count(",") + 1

	if ax is None: fig, ax = plt.subplots(figsize=(10, 6))
	else: fig = ax.figure

	bins = range(1, genre_counts.max() + 2)

	ax.hist(genre_counts, bins=bins, edgecolor="black", align="left")
	ax.set_xlabel("Number of Genres per Song")
	ax.set_ylabel("Number of Songs")
	ax.set_title("Genre Variety: How Many Genres Does Each Song Belong To?")
	ax.set_xticks(range(1, genre_counts.max() + 1))

	return fig


plot_genre_count_histogram(data)
plt.show()

### 3.5 Genre Distribution

How many songs belong to each of the top genres? Because genres are multilabel, a single song can contribute to multiple bars — the counts reflect genre *membership*, not exclusive assignment.

In [ ]:
def plot_genre_distribution(data: EDAData, n: int = 10,
	ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
	counts = data.genres_enc.sum().sort_values(ascending=True).tail(n)

	if ax is None: fig, ax = plt.subplots(figsize=(10, 6))
	else: fig = ax.figure

	ax.barh(counts.index, counts.values)
	ax.set_xlabel("Number of Songs")
	ax.set_title(f"Top {n} Genres by Song Count")

	return fig


plot_genre_distribution(data, n=10)
plt.show()

### 3.6 Lyrics Length Distribution

Some songs have dense, wordy lyrics while others are minimal. We look at three measures:
1. **Character count** — raw text length
2. **Word count** — total tokens after splitting on whitespace
3. **Meaningful word count** — words remaining after removing English stopwords (via NLTK)

This gives a sense of the corpus's text density and how much "signal" the lyrics embeddings have to work with.

In [ ]:
def plot_lyrics_length(data: EDAData,
	lang: str = "english",
) -> matplotlib.figure.Figure:
	lyrics = data.dataset["lyrics"]
	words = lyrics.str.split()

	stop = set(stopwords.words(lang))

	char_counts = lyrics.str.len()
	word_counts = words.str.len()
	meaningful_counts = words.apply(lambda ws: sum(1 for w in ws if w not in stop))

	fig, axes = plt.subplots(1, 3, figsize=(18, 5))

	axes[0].hist(char_counts, bins=50, edgecolor="black")
	axes[0].set_xlabel("Character Count")
	axes[0].set_ylabel("Number of Songs")
	axes[0].set_title("Lyrics Length (Characters)")

	axes[1].hist(word_counts, bins=50, edgecolor="black")
	axes[1].set_xlabel("Word Count")
	axes[1].set_title("Lyrics Length (Words)")

	axes[2].hist(meaningful_counts, bins=50, edgecolor="black")
	axes[2].set_xlabel("Meaningful Word Count")
	axes[2].set_title("Lyrics Length (Without Stopwords)")

	fig.tight_layout()

	return fig


plot_lyrics_length(data)
plt.show()

### 3.7 Sentiment Analysis by Genre

The assignment suggests using VADER for sentiment analysis. However, our lyrics are **already stemmed** — words like "happy" appear as "happi", "beautiful" as "beauti", etc. VADER relies on a hand-crafted lexicon of exact word forms, so stemmed text would largely miss its vocabulary.

> **Deviation**: We use **DistilBERT** via `transformers.pipeline("sentiment-analysis")` instead. DistilBERT uses BPE subword tokenization, which gracefully handles stemmed tokens by splitting them into recognized subwords. The model outputs a label (POSITIVE/NEGATIVE) and a confidence score, which we map to a [-1, +1] sentiment scale.

The violin plot below shows sentiment score distributions for each of the top-5 genres. We might expect pop songs to skew more positive and genres like rock or alternative to show broader distributions.

In [ ]:
def compute_sentiment(lyrics: pd.Series,
	batch_size: int = 64,
) -> pd.Series:
	device = 0 if torch.cuda.is_available() else -1

	classifier = pipeline("sentiment-analysis",  # type: ignore[call-overload]
		device=device,
		truncation=True,
		max_length=512,
	)

	results = classifier(lyrics.tolist(), batch_size=batch_size)

	scores = pd.Series(
		[r["score"] if r["label"] == "POSITIVE" else -r["score"] for r in results],
		index=lyrics.index,
		name="sentiment",
	)

	return scores


def plot_sentiment_by_genre(data: EDAData,
	top_k: int = 5,
) -> matplotlib.figure.Figure:
	sentiment = compute_sentiment(data.dataset["lyrics"])
	genres = top_genre_names(data, top_k)

	rows = []

	for genre in genres:
		mask = data.genres_enc[genre].astype(bool)

		for score in sentiment.loc[mask]:
			rows.append({"genre": genre, "sentiment": score})

	expanded = pd.DataFrame(rows)

	fig, ax = plt.subplots(figsize=(10, 6))

	sns.violinplot(data=expanded, x="genre", y="sentiment", ax=ax)

	ax.set_xlabel("Genre")
	ax.set_ylabel("Sentiment Score")
	ax.set_title("Sentiment Distribution by Genre (DistilBERT)")
	ax.axhline(0, color="gray", linestyle="--", alpha=0.5)

	return fig


plot_sentiment_by_genre(data, top_k=K)
plt.show()

### 3.8 Similarity Analysis

Given a query song, we find its top-5 most similar songs by **cosine similarity** on two separate embedding spaces:
- **Lyrics similarity** — songs with thematically similar text
- **Audio similarity** — songs with similar acoustic profiles (MFCC-derived)

The two modalities often disagree: a song may sound similar to others in a completely different genre but share lyrical themes with songs from yet another genre. We showcase several examples from different genres to illustrate this.

In [ ]:
def find_similar_songs(data: EDAData, song_id: str,
	k: int = 5,
	modality: str = "both",
) -> pd.DataFrame:
	results = {}

	if modality in ("lyrics", "both"):
		query = data.lyrics_emb.loc[[song_id]]
		sims = cosine_similarity(query, data.lyrics_emb)[0]
		sims = pd.Series(sims, index=data.lyrics_emb.index, name="lyrics_sim")
		sims = sims.drop(song_id).sort_values(ascending=False).head(k)
		results["lyrics_sim"] = sims

	if modality in ("audio", "both"):
		query = data.audio_emb.loc[[song_id]]
		sims = cosine_similarity(query, data.audio_emb)[0]
		sims = pd.Series(sims, index=data.audio_emb.index, name="audio_sim")
		sims = sims.drop(song_id).sort_values(ascending=False).head(k)
		results["audio_sim"] = sims

	all_ids = set()

	for s in results.values():
		all_ids.update(s.index)

	info = data.dataset.loc[list(all_ids), ["song", "artist", "genres"]]

	for name, sims in results.items():
		info[name] = sims

	return info.sort_values(
		by=list(results.keys()),
		ascending=False,
	)


def display_similarity_results(data: EDAData, song_id: str,
	k: int = 5,
) -> None:
	song = data.dataset.loc[song_id]

	print(f"\nQuery: {song['artist']} — {song['song']} [{song['genres']}]\n")

	for modality in ("lyrics", "audio"):
		result = find_similar_songs(data, song_id, k, modality)

		print(f"Top {k} by {modality} similarity:")
		print(result[["song", "artist", "genres", f"{modality}_sim"]].to_string())
		print()

In [ ]:
# Showcase similarity search with songs from different genres
example_ids = [
	"9epP1yOXfLKlkR3S",  # Carolina Liar — I'm Not Over [rock, indie rock]
	"MtxPSiYt0J3CTojA",  # Whitney Houston — You're Still My Man [pop, soul]
	"LQ5D2Q5SE914jB8V",  # Yuno — Grapefruit [electronic]
	"RLGHhOE3qeWQUFey",  # Gin Blossoms — Allison Road [alternative rock, rock]
]

for song_id in example_ids:
	display_similarity_results(data, song_id)
	print("=" * 80)

## Observations

**Dataset**: By treating genres as multilabel, we retain ~54,000 songs — significantly more than the ~10,000 expected from a multiclass approach. Most songs belong to 1–3 genres, confirming that multilabel is the natural representation.

**Embeddings**: The audio autoencoder compresses 104 MFCC features down to 11 dimensions, while the sentence transformer produces 384-dimensional lyrics vectors. Both are compact representations chosen for convenience — larger models and wider bottlenecks would likely improve downstream tasks.

**t-SNE**: The audio embeddings show some genre-level clustering (electronic songs tend to group together), but significant overlap remains — unsurprising given the low-dimensional bottleneck. Lyrics embeddings show less distinct clusters, suggesting that thematic content alone is a weaker genre discriminator than acoustic features.

**Sentiment**: DistilBERT sentiment distributions are fairly similar across genres, with most genres showing a slight positive skew. This is consistent with the observation that pop lyrics tend toward positive sentiment, while rock and alternative show broader variance.

**Similarity**: Audio and lyrics similarity often retrieve very different neighbors for the same query — a song's closest acoustic match may come from a completely different genre than its closest thematic match. This motivates multimodal fusion for downstream classification tasks.

## Part B — Multi-label Classification and Fusion

For Part B we keep the multilabel representation: every song is mapped to a set of genre labels, encoded as binary switches. The classification pipeline compares text-only, audio-only, concatenated early fusion, late fusion, bilinear pooling, and K-Means clustering analysis.

**Standard early fusion** concatenates the lyric and audio embeddings. **Bilinear pooling**, also called **outer-product fusion**, instead forms every pairwise text-audio product `text_i * audio_j` and uses only these cross-modal interaction features. With 384 lyric dimensions and 11 audio dimensions, the bilinear representation has `384 * 11 = 4224` features per song. This is not a correlation matrix; it is a per-song outer-product feature map.

The logistic models use fixed representation-specific regularization values: `C=10` for audio, `C=1` for text and concatenated early fusion, and `C=0.1` for bilinear pooling. The 10-fold cross-validation requested in the assignment is used to evaluate these fixed configurations, not to perform automatic hyperparameter tuning.

Runtime parallelism follows the source pipeline: text-only, audio-only, concatenated early fusion, and late fusion use fold-level `cross_validate(..., n_jobs=-1)`, while heavier evaluations are capped by `DEFAULT_HEAVY_CV_N_JOBS`. That cap is used for bilinear-pooling CV and for parallelizing independent K-Means cluster-count evaluations. The classifiers themselves remain single-job to avoid nested CPU oversubscription. Late fusion is implemented as a sklearn-compatible estimator that fits text and audio classifiers internally and averages their per-label probabilities.

The cell below is cache-aware: it displays saved CSV/PNG outputs from `results/` when they already exist, and otherwise runs the full intended Part B experiment, saves the artifacts, and displays the same tables and plots in the notebook. Set `FORCE_PART_B_RUN = True` in the cell to recompute everything.

### Part B Helper Implementation

The notebook embeds the Part B classification helpers here so it can be submitted and rerun without importing `src.classification`. The code mirrors the source module except for the command-line interface.

In [ ]:
# Self-contained Part B implementation.
# Synchronized from project/src/classification.py; CLI-only code is omitted.

from __future__ import annotations


import argparse
import importlib
import pathlib
import sys
from contextlib import contextmanager
from dataclasses import dataclass
from collections.abc import Iterable, Iterator
from typing import TYPE_CHECKING, Any, Protocol, cast

import numpy
import pandas

from joblib import Parallel, delayed, parallel_backend
from sklearn.base import BaseEstimator, ClassifierMixin, TransformerMixin, clone
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
	accuracy_score,
	adjusted_rand_score,
	f1_score,
	hamming_loss,
	jaccard_score,
	multilabel_confusion_matrix,
	precision_score,
	recall_score,
	silhouette_score,
)
from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


DEFAULT_TEXT_C = 1.
DEFAULT_AUDIO_C = 10.
DEFAULT_EARLY_C = 1.
DEFAULT_BILINEAR_C = 0.1

DEFAULT_CV_N_JOBS = 10
DEFAULT_HEAVY_CV_N_JOBS = 10


if TYPE_CHECKING:
	import matplotlib.axes
	import matplotlib.figure


class SupportsPredictProba(Protocol):
	def predict_proba(self, x: numpy.ndarray) -> numpy.ndarray | list[numpy.ndarray]:
		...


class BilinearPooling(TransformerMixin, BaseEstimator):
	def __init__(self, n_text_features: int) -> None:
		self.n_text_features = n_text_features
		self.text_scaler_: StandardScaler | None = None
		self.audio_scaler_: StandardScaler | None = None

	def fit(self, x: numpy.ndarray,
		y: numpy.ndarray | None = None,
	) -> BilinearPooling:
		del y

		text, audio = self.split(x)
		self.text_scaler_ = StandardScaler().fit(text)
		self.audio_scaler_ = StandardScaler().fit(audio)

		return self

	def transform(self, x: numpy.ndarray) -> numpy.ndarray:
		text_scaler = self.text_scaler_
		audio_scaler = self.audio_scaler_

		if text_scaler is None or audio_scaler is None:
			raise RuntimeError("BilinearPooling must be fitted before transform.")

		text, audio = self.split(x)
		text = numpy.asarray(text_scaler.transform(text), dtype = float)
		audio = numpy.asarray(audio_scaler.transform(audio), dtype = float)

		return (text[:, :, numpy.newaxis] * audio[:, numpy.newaxis, :]).reshape(len(x), -1)

	def split(self, x: numpy.ndarray) -> tuple[numpy.ndarray, numpy.ndarray]:
		values = numpy.asarray(x, dtype = float)

		return (
			values[:, :self.n_text_features],
			values[:, self.n_text_features:],
		)


class LateFusionClassifier(ClassifierMixin, BaseEstimator):
	def __init__(self,
		text_estimator: BaseEstimator,
		audio_estimator: BaseEstimator,
		n_text_features: int,
	) -> None:
		self.text_estimator = text_estimator
		self.audio_estimator = audio_estimator
		self.n_text_features = n_text_features
		self.text_model_: BaseEstimator | None = None
		self.audio_model_: BaseEstimator | None = None
		self.n_labels_: int | None = None
		self.classes_: numpy.ndarray | None = None

	def fit(self, x: numpy.ndarray,
		y: numpy.ndarray,
	) -> LateFusionClassifier:
		text, audio = self.split(x)
		self.text_model_ = cast(BaseEstimator, cast(Any, clone(self.text_estimator)).fit(text, y))
		self.audio_model_ = cast(BaseEstimator, cast(Any, clone(self.audio_estimator)).fit(audio, y))
		self.n_labels_ = y.shape[1]
		self.classes_ = numpy.arange(y.shape[1])

		return self

	def predict_proba(self, x: numpy.ndarray) -> numpy.ndarray:
		text_model = self.text_model_
		audio_model = self.audio_model_
		n_labels = self.n_labels_

		if text_model is None or audio_model is None or n_labels is None:
			raise RuntimeError("LateFusionClassifier must be fitted before predict_proba.")

		text, audio = self.split(x)

		text_proba = multilabel_proba(text_model, text, n_labels)
		audio_proba = multilabel_proba(audio_model, audio, n_labels)

		return (text_proba + audio_proba) / 2

	def predict(self, x: numpy.ndarray) -> numpy.ndarray:
		return probabilities_to_labels(self.predict_proba(x))

	def split(self, x: numpy.ndarray) -> tuple[numpy.ndarray, numpy.ndarray]:
		values = numpy.asarray(x, dtype = float)

		return (
			values[:, :self.n_text_features],
			values[:, self.n_text_features:],
		)


@dataclass
class MultilabelData:
	dataset: pandas.DataFrame
	labels: list[str]
	y: pandas.DataFrame
	text: pandas.DataFrame
	audio: pandas.DataFrame
	fused: pandas.DataFrame


@dataclass
class PredictionResult:
	name: str
	y_pred: pandas.DataFrame
	y_proba: pandas.DataFrame
	metrics: pandas.Series


@dataclass
class ExperimentResults:
	data: MultilabelData
	predictions: dict[str, PredictionResult]
	metrics: pandas.DataFrame
	cv_scores: pandas.DataFrame
	cv_summary: pandas.DataFrame
	confusion_matrices: dict[str, dict[str, pandas.DataFrame]]
	clustering: pandas.DataFrame | None = None


@contextmanager
def optional_progress(enabled: bool,
	total_steps: int,
) -> Iterator[tuple[Any, Any]]:
	if not enabled:
		yield None, None
		return

	try:
		rich_progress = importlib.import_module("rich.progress")
	except ModuleNotFoundError:
		print("rich is not installed; continuing without progress bars.", file = sys.stderr)
		yield None, None
		return

	bar_column = getattr(rich_progress, "BarColumn")
	progress_type = getattr(rich_progress, "Progress")
	spinner_column = getattr(rich_progress, "SpinnerColumn")
	text_column = getattr(rich_progress, "TextColumn")
	time_elapsed_column = getattr(rich_progress, "TimeElapsedColumn")

	with progress_type(
		spinner_column(),
		text_column("[progress.description]{task.description}"),
		bar_column(),
		time_elapsed_column(),
	) as progress:
		overall_task = progress.add_task("Overall classification pipeline", total = total_steps)
		yield progress, overall_task


@contextmanager
def progress_step(progress: Any,
	overall_task: Any,
	description: str,
) -> Iterator[None]:
	if progress is None or overall_task is None:
		yield
		return

	task = progress.add_task(description, total = None)
	completed = False

	try:
		yield
		completed = True
	finally:
		progress.remove_task(task)

		if completed:
			progress.advance(overall_task)


def load_multilabel_data(data_dir: str | pathlib.Path,
	k: int = 5,
	label_count: int | None = None,
) -> MultilabelData:
	data_dir = pathlib.Path(data_dir)
	label_count = label_count or k

	dataset = pandas.read_csv(data_dir / f"dataset.{k}.csv", index_col = 0)
	genres = pandas.read_parquet(data_dir / f"dataset.{k}.genres.parquet")
	audio = pandas.read_parquet(data_dir / f"dataset.{k}.audio.parquet")
	text = pandas.read_parquet(data_dir / f"dataset.{k}.lyrics.parquet")

	index = dataset.index.intersection(genres.index).intersection(audio.index).intersection(text.index)
	labels = genres.loc[index].sum().sort_values(ascending = False).head(label_count).index.tolist()
	y = genres.loc[index, labels].astype(int)
	mask = y.sum(axis = "columns") > 0

	dataset = dataset.loc[index].loc[mask]
	y = y.loc[mask]
	text = text.loc[index].loc[mask]
	audio = audio.loc[index].loc[mask]

	fused = pandas.concat(
		[
			text.add_prefix("text__"),
			audio.add_prefix("audio__"),
		],
		axis = "columns",
	)

	return MultilabelData(
		dataset = dataset,
		labels = labels,
		y = y,
		text = text,
		audio = audio,
		fused = fused,
	)


def sample_data(data: MultilabelData,
	max_samples: int | None,
	random_state: int = 42,
) -> MultilabelData:
	if max_samples is None or max_samples >= len(data.y):
		return data

	index = data.y.sample(n = max_samples, random_state = random_state).index

	return MultilabelData(
		dataset = data.dataset.loc[index],
		labels = data.labels,
		y = data.y.loc[index],
		text = data.text.loc[index],
		audio = data.audio.loc[index],
		fused = data.fused.loc[index],
	)


def build_classifier(kind: str = "logistic",
	random_state: int = 42,
	regularization_c: float = 1.,
) -> BaseEstimator:
	if kind == "logistic":
		base = LogisticRegression(
			C = regularization_c,
			max_iter = 1000,
			class_weight = "balanced",
			solver = "liblinear",
			random_state = random_state,
		)

		return make_pipeline(
			StandardScaler(),
			OneVsRestClassifier(base, n_jobs = 1),
		)

	if kind == "random_forest":
		base = RandomForestClassifier(
			n_estimators = 300,
			class_weight = "balanced_subsample",
			n_jobs = 1,
			random_state = random_state,
		)

		return OneVsRestClassifier(base, n_jobs = 1)

	raise ValueError(f"Unsupported classifier kind: {kind}")


def build_bilinear_classifier(kind: str,
	n_text_features: int,
	random_state: int = 42,
	regularization_c: float = 1.,
) -> BaseEstimator:
	if kind == "logistic":
		base = LogisticRegression(
			C = regularization_c,
			max_iter = 1000,
			class_weight = "balanced",
			solver = "liblinear",
			random_state = random_state,
		)

		return make_pipeline(
			BilinearPooling(n_text_features),
			OneVsRestClassifier(base, n_jobs = 1),
		)

	if kind == "random_forest":
		base = RandomForestClassifier(
			n_estimators = 300,
			class_weight = "balanced_subsample",
			n_jobs = 1,
			random_state = random_state,
		)

		return make_pipeline(
			BilinearPooling(n_text_features),
			OneVsRestClassifier(base, n_jobs = 1),
		)

	raise ValueError(f"Unsupported classifier kind: {kind}")


def build_late_fusion_classifier(kind: str,
	n_text_features: int,
	random_state: int = 42,
	text_regularization_c: float = DEFAULT_TEXT_C,
	audio_regularization_c: float = DEFAULT_AUDIO_C,
) -> BaseEstimator:
	return LateFusionClassifier(
		text_estimator = build_classifier(
			kind,
			random_state = random_state,
			regularization_c = text_regularization_c,
		),
		audio_estimator = build_classifier(
			kind,
			random_state = random_state,
			regularization_c = audio_regularization_c,
		),
		n_text_features = n_text_features,
	)


def labelset_codes(y: pandas.DataFrame | numpy.ndarray) -> numpy.ndarray:
	if isinstance(y, pandas.DataFrame):
		values = y.to_numpy(dtype = int)
	else:
		values = numpy.asarray(y, dtype = int)

	return numpy.array(["".join(row.astype(str)) for row in values])


def make_cv_splits(y: pandas.DataFrame,
	n_splits: int = 10,
	random_state: int = 42,
) -> list[tuple[numpy.ndarray, numpy.ndarray]]:
	codes = pandas.Series(labelset_codes(y), index = y.index)

	if codes.value_counts().min() >= n_splits:
		splitter = StratifiedKFold(
			n_splits = n_splits,
			shuffle = True,
			random_state = random_state,
		)

		return list(splitter.split(numpy.zeros(len(y)), codes))

	splitter = KFold(
		n_splits = n_splits,
		shuffle = True,
		random_state = random_state,
	)

	return list(splitter.split(numpy.zeros(len(y))))


def multilabel_proba(estimator: BaseEstimator,
	x: numpy.ndarray,
	n_labels: int,
) -> numpy.ndarray:
	raw = cast(SupportsPredictProba, estimator).predict_proba(x)

	if isinstance(raw, list):
		classes = getattr(estimator, "classes_", [None] * len(raw))
		columns = []

		for probabilities, label_classes in zip(raw, classes):
			if label_classes is None:
				columns.append(probabilities[:, -1])
				continue

			positive = numpy.flatnonzero(numpy.asarray(label_classes) == 1)
			if len(positive) == 0:
				columns.append(numpy.zeros(len(x)))
			else:
				columns.append(probabilities[:, positive[0]])

		return numpy.column_stack(columns)

	probabilities = numpy.asarray(raw)

	if probabilities.ndim == 3:
		return probabilities[:, :, -1]

	if probabilities.shape[1] != n_labels:
		raise ValueError(
			f"Expected {n_labels} probability columns, got {probabilities.shape[1]}."
		)

	return probabilities


def probabilities_to_labels(probabilities: numpy.ndarray,
	threshold: float = 0.5,
	ensure_one: bool = True,
) -> numpy.ndarray:
	y_pred = (probabilities >= threshold).astype(int)

	if ensure_one:
		empty = y_pred.sum(axis = 1) == 0
		y_pred[empty, probabilities[empty].argmax(axis = 1)] = 1

	return y_pred


def score_multilabel(y_true: pandas.DataFrame | numpy.ndarray,
	y_pred: pandas.DataFrame | numpy.ndarray,
) -> pandas.Series:
	zero_division = cast(Any, 0)

	return pandas.Series({
		"subset_accuracy": accuracy_score(y_true, y_pred),
		"hamming_loss": hamming_loss(y_true, y_pred),
		"precision_macro": precision_score(y_true, y_pred, average = "macro", zero_division = zero_division),
		"recall_macro": recall_score(y_true, y_pred, average = "macro", zero_division = zero_division),
		"f1_macro": f1_score(y_true, y_pred, average = "macro", zero_division = zero_division),
		"precision_micro": precision_score(y_true, y_pred, average = "micro", zero_division = zero_division),
		"recall_micro": recall_score(y_true, y_pred, average = "micro", zero_division = zero_division),
		"f1_micro": f1_score(y_true, y_pred, average = "micro", zero_division = zero_division),
		"f1_samples": f1_score(y_true, y_pred, average = "samples", zero_division = zero_division),
		"jaccard_samples": jaccard_score(y_true, y_pred, average = "samples", zero_division = zero_division),
	})


def make_prediction_result(name: str,
	y_true: pandas.DataFrame,
	y_pred: numpy.ndarray,
	y_proba: numpy.ndarray,
) -> PredictionResult:
	y_pred_frame = pandas.DataFrame(
		y_pred,
		index = y_true.index,
		columns = y_true.columns,
	)
	y_proba_frame = pandas.DataFrame(
		y_proba,
		index = y_true.index,
		columns = y_true.columns,
	)

	return PredictionResult(
		name = name,
		y_pred = y_pred_frame,
		y_proba = y_proba_frame,
		metrics = score_multilabel(y_true, y_pred_frame),
	)


def cross_validated_predictions(name: str,
	estimator: BaseEstimator,
	x: pandas.DataFrame,
	y: pandas.DataFrame,
	splits: Iterable[tuple[numpy.ndarray, numpy.ndarray]],
	threshold: float = 0.5,
	n_jobs: int = DEFAULT_CV_N_JOBS,
) -> tuple[PredictionResult, pandas.DataFrame]:
	x_values = x.to_numpy(dtype = float)
	y_values = y.to_numpy(dtype = int)
	split_list = list(splits)
	pre_dispatch = n_jobs if n_jobs not in (None, -1) else "2*n_jobs"
	cv_result = cross_validate(
		estimator,
		x_values,
		y_values,
		cv = split_list,
		n_jobs = n_jobs,
		pre_dispatch = cast(Any, pre_dispatch),
		return_estimator = True,
		return_indices = True,  # type: ignore
	)

	probabilities = numpy.zeros(y_values.shape, dtype = float)
	fold_rows = []

	for fold, (model, test_idx) in enumerate(
		zip(cv_result["estimator"], cv_result["indices"]["test"]),
		start = 1,
	):
		fold_probabilities = multilabel_proba(model, x_values[test_idx], y.shape[1])
		fold_predictions = probabilities_to_labels(fold_probabilities, threshold = threshold)
		fold_scores = score_multilabel(y_values[test_idx], fold_predictions)

		probabilities[test_idx] = fold_probabilities

		fold_rows.append({
			"model": name,
			"fold": fold,
			"fit_time": cv_result["fit_time"][fold - 1],
			"score_time": cv_result["score_time"][fold - 1],
			**fold_scores.to_dict(),
		})

	predictions = probabilities_to_labels(probabilities, threshold = threshold)
	fold_frame = pandas.DataFrame(fold_rows).set_index(["model", "fold"])

	return make_prediction_result(name, y, predictions, probabilities), fold_frame


def confusion_by_label(y_true: pandas.DataFrame,
	y_pred: pandas.DataFrame,
) -> dict[str, pandas.DataFrame]:
	matrices = multilabel_confusion_matrix(y_true, y_pred)

	return {
		label: pandas.DataFrame(
			matrix,
			index = ["actual_0", "actual_1"],
			columns = ["pred_0", "pred_1"],
		)
		for label, matrix in zip(y_true.columns, matrices)
	}


def evaluate_kmeans_count(n_clusters: int,
	x: numpy.ndarray,
	y_values: numpy.ndarray,
	labelsets: numpy.ndarray,
	sample_size: int | None,
	random_state: int,
) -> dict[str, float | int]:
	clusters = KMeans(
		n_clusters = n_clusters,
		random_state = random_state,
		n_init = cast(Any, 10),
	).fit_predict(x)

	per_label_ari = [
		adjusted_rand_score(y_values[:, label], clusters)
		for label in range(y_values.shape[1])
	]

	if sample_size is not None and sample_size < len(x):
		silhouette = silhouette_score(
			x,
			clusters,
			sample_size = sample_size,
			random_state = random_state,
		)
	else:
		silhouette = silhouette_score(x, clusters)

	return {
		"n_clusters": n_clusters,
		"silhouette": silhouette,  # type: ignore
		"ari_labelset": adjusted_rand_score(labelsets, clusters),
		"ari_per_label_macro": float(numpy.mean(per_label_ari)),
	}


def evaluate_kmeans(data: MultilabelData,
	cluster_counts: Iterable[int] = range(2, 16),
	sample_size: int | None = 10000,
	random_state: int = 8312,
	n_jobs: int = DEFAULT_HEAVY_CV_N_JOBS,
) -> pandas.DataFrame:
	x = StandardScaler().fit_transform(data.fused.to_numpy(dtype = float))
	y_values = data.y.to_numpy(dtype = int)
	labelsets = labelset_codes(y_values)
	cluster_count_list = list(cluster_counts)
	pre_dispatch = n_jobs if n_jobs not in (None, -1) else "2*n_jobs"

	with parallel_backend("loky", inner_max_num_threads = 1):
		rows = Parallel(n_jobs = n_jobs, pre_dispatch = cast(Any, pre_dispatch))(
			delayed(evaluate_kmeans_count)(
				n_clusters,
				x,
				y_values,
				labelsets,
				sample_size,
				random_state,
			)
			for n_clusters in cluster_count_list
		)

	return pandas.DataFrame(rows).set_index("n_clusters")  # type: ignore


def run_experiments(data_dir: str | pathlib.Path,
	k: int = 5,
	label_count: int | None = None,
	n_splits: int = 10,
	classifier: str = "logistic",
	regularization_c: float | None = None,
	text_regularization_c: float = DEFAULT_TEXT_C,
	audio_regularization_c: float = DEFAULT_AUDIO_C,
	early_regularization_c: float = DEFAULT_EARLY_C,
	bilinear_regularization_c: float = DEFAULT_BILINEAR_C,
	threshold: float = 0.5,
	max_samples: int | None = None,
	include_bilinear: bool = True,
	include_clustering: bool = True,
	random_state: int = 42,
	show_progress: bool = False,
	heavy_n_jobs: int = DEFAULT_HEAVY_CV_N_JOBS,
) -> ExperimentResults:
	if regularization_c is not None:
		text_regularization_c = regularization_c
		audio_regularization_c = regularization_c
		early_regularization_c = regularization_c
		bilinear_regularization_c = regularization_c

	progress_steps = 6 + int(include_bilinear) + int(include_clustering)

	with optional_progress(show_progress, progress_steps) as (progress, overall_task):
		with progress_step(progress, overall_task, "Loading cached Part B data"):
			data = load_multilabel_data(data_dir, k = k, label_count = label_count)

		with progress_step(progress, overall_task, f"Preparing {n_splits}-fold CV splits"):
			data = sample_data(data, max_samples = max_samples, random_state = random_state)
			splits = make_cv_splits(data.y, n_splits = n_splits, random_state = random_state)

		with progress_step(progress, overall_task, f"Text-only CV ({n_splits} folds, n_jobs={DEFAULT_CV_N_JOBS})"):
			text, text_cv = cross_validated_predictions(
				"Text-only",
				build_classifier(classifier, random_state = random_state, regularization_c = text_regularization_c),
				data.text,
				data.y,
				splits,
				threshold = threshold,
				n_jobs = DEFAULT_CV_N_JOBS,
			)

		with progress_step(progress, overall_task, f"Audio-only CV ({n_splits} folds, n_jobs={DEFAULT_CV_N_JOBS})"):
			audio, audio_cv = cross_validated_predictions(
				"Audio-only",
				build_classifier(classifier, random_state = random_state, regularization_c = audio_regularization_c),
				data.audio,
				data.y,
				splits,
				threshold = threshold,
				n_jobs = DEFAULT_CV_N_JOBS,
			)

		with progress_step(
			progress,
			overall_task,
			f"Early-fusion CV ({n_splits} folds, n_jobs={DEFAULT_CV_N_JOBS})",
		):
			early, early_cv = cross_validated_predictions(
				"Early fusion",
				build_classifier(classifier, random_state = random_state, regularization_c = early_regularization_c),
				data.fused,
				data.y,
				splits,
				threshold = threshold,
				n_jobs = DEFAULT_CV_N_JOBS,
			)

		with progress_step(
			progress,
			overall_task,
			f"Late-fusion CV ({n_splits} folds, n_jobs={DEFAULT_CV_N_JOBS})",
		):
			late, late_cv = cross_validated_predictions(
				"Late fusion",
				build_late_fusion_classifier(
					classifier,
					n_text_features = data.text.shape[1],
					random_state = random_state,
					text_regularization_c = text_regularization_c,
					audio_regularization_c = audio_regularization_c,
				),
				data.fused,
				data.y,
				splits,
				threshold = threshold,
				n_jobs = DEFAULT_CV_N_JOBS,
			)

		bilinear = bilinear_cv = None

		if include_bilinear:
			with progress_step(
				progress,
				overall_task,
				f"Bilinear-pooling CV ({n_splits} folds, n_jobs={heavy_n_jobs})",
			):
				bilinear, bilinear_cv = cross_validated_predictions(
					"Bilinear pooling",
					build_bilinear_classifier(
						classifier,
						n_text_features = data.text.shape[1],
						random_state = random_state,
						regularization_c = bilinear_regularization_c,
					),
					data.fused,
					data.y,
					splits,
					threshold = threshold,
					n_jobs = heavy_n_jobs,
				)

		clustering = None

		if include_clustering:
			with progress_step(progress, overall_task, f"K-Means clustering evaluation (n_jobs={heavy_n_jobs})"):
				clustering = evaluate_kmeans(data, n_jobs = heavy_n_jobs)

	predictions = {
		result.name: result
		for result in (text, audio, early, late, bilinear)
		if result is not None
	}
	metrics = pandas.DataFrame(
		{
			name: result.metrics
			for name, result in predictions.items()
		}
	).T
	cv_scores = pandas.concat(
		[
			frame
			for frame in (text_cv, audio_cv, early_cv, late_cv, bilinear_cv)
			if frame is not None
		],
		axis = "index",
	)
	cv_summary = cast(pandas.DataFrame, cv_scores.groupby(level = "model").agg(["mean", "std"]))
	confusions = {
		name: confusion_by_label(data.y, result.y_pred)
		for name, result in predictions.items()
	}
	return ExperimentResults(
		data = data,
		predictions = predictions,
		metrics = metrics,
		cv_scores = cv_scores,
		cv_summary = cv_summary,
		confusion_matrices = confusions,
		clustering = clustering,
	)


def plot_f1_comparison(metrics: pandas.DataFrame,
	ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
	import matplotlib.pyplot

	if ax is None:
		fig, ax = matplotlib.pyplot.subplots(figsize = (8, 5))
	else:
		fig = ax.figure

	metrics["f1_macro"].plot(kind = "bar", ax = ax, color = "#3b6ea8")
	ax.set_ylabel("Macro F1")
	ax.set_xlabel("")
	ax.set_ylim(0, 1)
	ax.set_title("Multi-label Genre Classification")
	ax.tick_params(axis = "x", rotation = 20)

	return fig  # type: ignore


def plot_label_confusions(confusions: dict[str, pandas.DataFrame],
	title: str,
) -> matplotlib.figure.Figure:
	import matplotlib.pyplot

	n_labels = len(confusions)
	fig, axes = matplotlib.pyplot.subplots(
		1,
		n_labels,
		figsize = (3.2 * n_labels, 3.2),
	)
	axes = numpy.atleast_1d(axes)

	for ax, (label, matrix) in zip(axes, confusions.items()):
		ax.imshow(matrix.values, cmap = "Blues")
		ax.set_title(label)
		ax.set_xticks([0, 1], ["pred 0", "pred 1"])
		ax.set_yticks([0, 1], ["actual 0", "actual 1"])

		for row in range(2):
			for column in range(2):
				ax.text(
					column,
					row,
					f"{matrix.iloc[row, column]:,}",
					ha = "center",
					va = "center",
					color = "black",
				)

	fig.suptitle(title)
	fig.tight_layout()

	return fig


def plot_clustering_metrics(clustering: pandas.DataFrame,
	ax: matplotlib.axes.Axes | None = None,
) -> matplotlib.figure.Figure:
	import matplotlib.pyplot

	if ax is None:
		fig, ax = matplotlib.pyplot.subplots(figsize = (8, 5))
	else:
		fig = ax.figure

	clustering[["silhouette", "ari_labelset", "ari_per_label_macro"]].plot(
		ax = ax,
		marker = "o",
	)
	ax.set_xlabel("Number of clusters")
	ax.set_ylabel("Score")
	ax.set_title("K-Means Clustering Evaluation")
	ax.grid(alpha = 0.25)

	return fig  # type: ignore


def safe_filename(name: str) -> str:
	slug = "".join(
		char.lower() if char.isalnum() else "_"
		for char in name
	).strip("_")

	return "_".join(part for part in slug.split("_") if part)


def save_evaluation_outputs(results: ExperimentResults,
	output: pathlib.Path,
) -> None:
	import os

	output.mkdir(parents = True, exist_ok = True)
	os.environ.setdefault("MPLCONFIGDIR", str(output / ".matplotlib"))

	import matplotlib.pyplot

	results.metrics.to_csv(output / "classification_metrics.csv")
	results.cv_scores.to_csv(output / "classification_cv_folds.csv")
	results.cv_summary.to_csv(output / "classification_cv_summary.csv")

	fig = plot_f1_comparison(results.metrics)
	fig.savefig(output / "f1_macro_comparison.png", dpi = 150, bbox_inches = "tight")
	matplotlib.pyplot.close(fig)

	for model_name, confusions in results.confusion_matrices.items():
		fig = plot_label_confusions(confusions, f"{model_name} Confusion Matrices")
		fig.savefig(
			output / f"confusion_{safe_filename(model_name)}.png",
			dpi = 150,
			bbox_inches = "tight",
		)
		matplotlib.pyplot.close(fig)

	if results.clustering is not None:
		results.clustering.to_csv(output / "clustering_metrics.csv")

		fig = plot_clustering_metrics(results.clustering)
		fig.savefig(output / "clustering_metrics.png", dpi = 150, bbox_inches = "tight")
		matplotlib.pyplot.close(fig)

We run the experiments below:

In [ ]:
from IPython.display import Image, display

# Set this to True to recompute the full Part B experiment.
FORCE_PART_B_RUN = False
REQUIRED_RESULT_FILES = [
	"classification_metrics.csv",
	"classification_cv_summary.csv",
	"f1_macro_comparison.png",
]
RESULTS_LABEL = "results/"

print(f"Heavy evaluation n_jobs cap: {DEFAULT_HEAVY_CV_N_JOBS}")

cached_results_available = all(
	(RESULTS_DIR / filename).exists()
	for filename in REQUIRED_RESULT_FILES
)
results = None

if FORCE_PART_B_RUN or not cached_results_available:
	if not cached_results_available:
		print("Cached Part B outputs were not found; running the full experiment.")

	results = run_experiments(
		DATA_DIR,
		k=K,
		n_splits=10,
		show_progress=True,
		heavy_n_jobs=DEFAULT_HEAVY_CV_N_JOBS,
	)

	save_evaluation_outputs(results, RESULTS_DIR)
	print(f"Saved classification CSV and PNG outputs to {RESULTS_LABEL}")
	print("Labels:", ", ".join(results.data.labels))

	metrics = results.metrics
	cv_summary = results.cv_summary
	clustering = results.clustering
else:
	print(f"Loaded cached Part B outputs from {RESULTS_LABEL}")
	metrics = pd.read_csv(RESULTS_DIR / "classification_metrics.csv", index_col=0)
	cv_summary = pd.read_csv(
		RESULTS_DIR / "classification_cv_summary.csv",
		header=[0, 1],
		index_col=0,
	)
	clustering_path = RESULTS_DIR / "clustering_metrics.csv"
	clustering = pd.read_csv(clustering_path, index_col=0) if clustering_path.exists() else None

display(metrics.round(4))

summary_columns = [
	("precision_macro", "mean"),
	("recall_macro", "mean"),
	("f1_macro", "mean"),
	("f1_macro", "std"),
]
available_summary_columns = [column for column in summary_columns if column in cv_summary.columns]
display(cv_summary.loc[:, available_summary_columns].round(4))

if clustering is not None:
	display(clustering.round(4))

plot_paths = [
	RESULTS_DIR / "f1_macro_comparison.png",
	*sorted(RESULTS_DIR.glob("confusion_*.png")),
]
clustering_plot = RESULTS_DIR / "clustering_metrics.png"

if clustering_plot.exists():
	plot_paths.append(clustering_plot)

for plot_path in plot_paths:
	if plot_path.exists():
		print(plot_path.name)
		display(Image(filename=str(plot_path)))